# 04 · 讀懂 Optuna 的搜尋過程（視覺化）

> **這本為什麼是 notebook**：`optuna.visualization` 產出的是 Plotly 互動圖，
> 可以滑鼠移上去看每個 trial 的細節。這幾張圖的用途不是「好看」，
> 而是**決定下一輪要怎麼調搜尋範圍**——那是一個需要人看著圖判斷的動作。
>
> **配套腳本**：同資料夾的 `01`–`03` 維持 `.py`。特別是 `02_mlflow_callback.py`，
> 它把每個 trial 寫成一個 MLflow run；MLflow 的 run lifecycle 在 notebook 裡
> 很容易因為 cell 中途報錯而留下未關閉的 run，所以刻意不搬進來。
>
> **前置**：先跑過 `01_objective_basic.py`，理解 `objective` / `suggest` / `optimize` 三個動詞。

In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """從當前目錄往上找到 mlops-course 根目錄（含 datasets/ 的那層）。

    notebook 沒有 __file__，而且你可能從任何位置啟動 Jupyter，
    所以用「往上層找標記檔」定位，不寫死相對路徑。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "datasets" / "iris.csv").exists():
            return base
    raise FileNotFoundError("找不到 mlops-course/datasets/，請在 mlops-course/ 之內開啟本 notebook")


ROOT = find_course_root()
print("course root =", ROOT)

import optuna
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split

SEED = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)  # 關掉逐 trial 的 log，圖才是主角

df = pd.read_csv(ROOT / "datasets" / "iris.csv")
X = df.drop(columns=["target", "target_name"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"訓練集 {X_train.shape[0]} 筆、測試集 {X_test.shape[0]} 筆")

## 1. 跑一場搜尋

跟 `01_objective_basic.py` 同一個 `objective`，但 trial 數拉到 60——
**圖需要足夠的點才看得出型態**。

In [ ]:
def objective(trial: optuna.Trial) -> float:
    C = trial.suggest_float("C", 1e-3, 1e2, log=True)
    max_iter = trial.suggest_int("max_iter", 100, 500, step=100)
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])

    model = LogisticRegression(C=C, max_iter=max_iter, solver=solver, random_state=SEED)
    return float(cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean())


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),  # 固定 seed → 這本 notebook 可重現
)
study.optimize(objective, n_trials=60)

print(f"best_value  = {study.best_value:.4f}")
print(f"best_params = {study.best_params}")

In [ ]:
# trials 也可以直接轉成 DataFrame，做任何你熟悉的 pandas 操作
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
trials_df.sort_values("value", ascending=False).head(5)

## 2. 收斂歷史：什麼時候可以停？

橫軸是 trial 編號，紅線是「到目前為止的最佳值」。
**紅線走平代表再花錢也買不到更好的結果**——這就是你決定停止的依據。

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig

## 3. 超參重要性：下一輪該調哪一個？

這張圖回答「我的分數主要被誰決定」。
**重要性低的超參可以直接固定住**，把預算集中在重要的那個上面。

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig

## 4. 平行座標：好的 trial 集中在哪個區間？

每條線是一個 trial，顏色越深分數越高。
**看深色線在每個軸上穿過哪一段**——那就是你下一輪該縮小到的範圍。

In [ ]:
fig = optuna.visualization.plot_parallel_coordinate(study, params=["C", "max_iter"])
fig

## 5. Slice plot：單一超參的響應曲線

把其他超參的影響攤平，只看一個超參跟分數的關係。
**如果最佳點貼在搜尋範圍的邊界，代表範圍設太窄了，該往外擴。**

In [ ]:
fig = optuna.visualization.plot_slice(study, params=["C", "max_iter"])
fig

## 6. 把圖讀成決策

這四張圖各自對應一個具體動作：

| 圖 | 它回答的問題 | 你該做的事 |
| :--- | :--- | :--- |
| optimization history | 還有沒有進步空間？ | 紅線走平就停，別再燒預算 |
| param importances | 誰在決定分數？ | 不重要的超參固定住，省下的 trial 給重要的 |
| parallel coordinate | 好的 trial 長什麼樣？ | 把範圍縮到深色線集中的區間，再跑一輪 |
| slice plot | 範圍設得對嗎？ | 最佳點貼邊界 → 往外擴 |

**接下來換介質**：你已經知道要搜什麼範圍了。
下一步是「讓這些 trial 被記錄下來、可以被比較與追溯」——那是 MLflow 的工作，
而且它要能在 CI 裡無人值守地跑，所以是腳本：

```bash
python 02_mlflow_callback.py   # 每個 trial = 一個 MLflow run
mlflow ui                      # 展開 parent run 看 20 個 child run
```